# FL-for-Aircraft on Kaggle T4×2 — smoke test + GPU campaigns

Runs on **Kaggle's Jupyter Server** (connected from VS Code).

**Sections 1–5** validate the pipeline (GPU visible, clone, data, deps, forward
pass). **Sections 6–9** run the reviewer-proofing campaigns on the GPU:

- **Exp 2** — N=6 (FD001+FD003, 3+3 clients), Krum-f2 now valid
- **Exp 3** — FD002+FD004 (19 features, auto trigger index), the heavy run
- aggregate seeds → mean ± std, then zip `results/` for download

> Cells execute on **Kaggle**, not your laptop. We do NOT run `pip install -e .`
> (pyproject pins Python 3.12; Kaggle may differ) — adding `src/` to `sys.path`
> sidesteps that. **Read Section 6 first: restarting the session wipes
> everything, so download your results before you stop.**


## 1. Environment + GPU check

In [1]:
import platform
import torch

print(f"Python      : {platform.python_version()}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA build  : {torch.version.cuda}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count   : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  cuda:{i}    : {torch.cuda.get_device_name(i)}")
else:
    print("WARNING: no GPU visible - set Accelerator to GPU T4 x2 in Notebook Settings")

Python      : 3.12.13
PyTorch     : 2.10.0+cu128
CUDA build  : 12.8
CUDA avail  : True
GPU count   : 2
  cuda:0    : Tesla T4
  cuda:1    : Tesla T4


## 2. Clone the multiseed branch (anonymous - no token)

In [2]:
import os

REPO_DIR = "/kaggle/working/FL-for-Aircraft"
BRANCH = "multiseed"
if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} --depth 1 https://github.com/Chinmoy17/FL-for-Aircraft.git {REPO_DIR}
else:
    print("Repo already cloned; pulling latest")
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}
!git log --oneline -1

Cloning into '/kaggle/working/FL-for-Aircraft'...
remote: Enumerating objects: 617, done.
remote: Counting objects: 100% (617/617), done.
remote: Compressing objects: 100% (555/555), done.
remote: Total 617 (delta 78), reused 494 (delta 56), pack-reused 0 (from 0)
Receiving objects: 100% (617/617), 23.95 MiB | 27.97 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/kaggle/working/FL-for-Aircraft
2384fcd (grafted, HEAD -> multiseed, origin/multiseed) Enhance kaggle_run notebook and run_rq7_multiseed script with detailed instructions for GPU campaigns and new argument options


## 3. Confirm CMAPSS data came with the clone

In [3]:
!ls -la Dataset/CMAPSS_NASA/ | head -12

total 44332
drwxr-xr-x 2 root root     4096 Jul 27 08:37 .
drwxr-xr-x 3 root root     4096 Jul 27 08:37 ..
-rw-r--r-- 1 root root   434158 Jul 27 08:37 Damage Propagation Modeling.pdf
-rw-r--r-- 1 root root     2442 Jul 27 08:37 readme.txt
-rw-r--r-- 1 root root      429 Jul 27 08:37 RUL_FD001.txt
-rw-r--r-- 1 root root     1110 Jul 27 08:37 RUL_FD002.txt
-rw-r--r-- 1 root root      428 Jul 27 08:37 RUL_FD003.txt
-rw-r--r-- 1 root root     1084 Jul 27 08:37 RUL_FD004.txt
-rw-r--r-- 1 root root  2228855 Jul 27 08:37 test_FD001.txt
-rw-r--r-- 1 root root  5734587 Jul 27 08:37 test_FD002.txt
-rw-r--r-- 1 root root  2826651 Jul 27 08:37 test_FD003.txt


## 4. Install the one extra dependency

Kaggle already ships torch / numpy / pandas / scikit-learn / pyyaml / tqdm.
Only `captum` (used by the interpretability module) is extra. We skip
`pip install -e .` on purpose (Python-version pin).

In [4]:
!pip install captum -q
print("deps done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 25.1 MB/s eta 0:00:00
deps done


## 5. GPU forward-pass smoke test

Adds `src/` to the path, builds the multi-task CNN, moves it plus a dummy
batch to the GPU, and runs a forward pass. Proves the model is
GPU-compatible on Kaggle even before we wire device support into the
training loop.

In [5]:
import sys

SRC = "/kaggle/working/FL-for-Aircraft/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import torch
from fl_aircraft.models import MultiTaskCNN, MultiTaskCNNConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MultiTaskCNN(MultiTaskCNNConfig(n_features=17, window_size=30)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"model params : {n_params:,}")
print(f"device       : {device}")

x = torch.randn(8, 30, 17, device=device)
model.eval()
with torch.no_grad():
    pred = model(x)
print(f"forward OK   : rul={tuple(pred.rul.shape)}, on {pred.rul.device}")
print("\nSMOKE TEST PASSED" if str(pred.rul.device).startswith(device) else "device mismatch")

model params : 30,018
device       : cuda
forward OK   : rul=(8,), on cuda:0

SMOKE TEST PASSED


In [6]:
# Preflight — run this after any restart to confirm the repo, path, and GPU
# are ready before launching a campaign. Idempotent; safe to re-run.
import os, sys, torch

REPO = "/kaggle/working/FL-for-Aircraft"
if REPO + "/src" not in sys.path:
    sys.path.insert(0, REPO + "/src")

print("repo exists :", os.path.isdir(REPO))
print("cuda        :", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
!cd {REPO} && git log --oneline -1


repo exists : True
cuda        : True | GPUs: 2
2384fcd (grafted, HEAD -> multiseed, origin/multiseed) Enhance kaggle_run notebook and run_rq7_multiseed script with detailed instructions for GPU campaigns and new argument options


## 6. Before you run — restart & persistence (READ)

**Kaggle sessions are ephemeral.** When you stop/restart the session you get a
fresh container:

- `/kaggle/working` is wiped — the clone, `pip install captum`, **and any
  `results/` you produced**
- `sys.path` / installed packages / all variables reset
- the **VS Code Compatible URL changes** (new token) → re-copy it from the
  Kaggle *Jupyter Server* panel and reconnect the kernel

**So every restart:** *Run All* from the top (re-clone → install → path →
preflight), then run the experiment cells.

**Save results before stopping** — use the zip cell at the very bottom, or use
**Save Version → Save & Run All (Commit)** to run headless (up to the GPU time
limit) with `/kaggle/working` kept as a downloadable version output.

> The runner **auto-resumes** finished cells from per-round CSVs *within* a live
> session, but not across a restart (the working dir is gone).


In [11]:
# Campaign helpers.
#   run_campaign  = one GPU, seeds sequential, streams output live inline.
#   run_parallel  = split seeds across both T4s; DETACHED + non-blocking.
#   watch         = live auto-refresh of both GPU logs until done. Pressing ⏹
#                   stops the WATCH only — training keeps running because the
#                   workers are launched in their own session.
#   tail_logs     = one-shot peek at the last N lines of each GPU log.
#   wait_all      = block until the parallel processes finish.
import os, sys, time, subprocess
from IPython.display import clear_output

REPO = "/kaggle/working/FL-for-Aircraft"


def _logpath(out_root, gpu):
    return f"{REPO}/{out_root.replace('/', '_')}_{gpu.replace(':', '')}.log"


def run_campaign(subsets, n_per, out_root, seeds=(42, 43, 44, 45, 46),
                 device="cuda", skip_cells=None):
    cmd = [sys.executable, "scripts/run_rq7_multiseed.py",
           "--subsets", *subsets, "--n-clients-per-subset", str(n_per),
           "--seeds", *map(str, seeds), "--device", device,
           "--out-root", out_root]
    if skip_cells:
        cmd += ["--skip-cells", *skip_cells]
    print(">>>", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=REPO).returncode


def run_parallel(subsets, n_per, out_root, seeds=(42, 43, 44, 45, 46),
                 gpus=("cuda:0", "cuda:1"), skip_cells=None):
    """Split seeds across both T4s. DETACHED (start_new_session) so a cell
    interrupt can't kill training, and non-blocking so you can watch()."""
    os.makedirs(f"{REPO}/{out_root}", exist_ok=True)
    seeds = list(seeds)
    procs = []
    for i, gpu in enumerate(gpus):
        chunk = seeds[i::len(gpus)]        # round-robin split
        if not chunk:
            continue
        cmd = [sys.executable, "scripts/run_rq7_multiseed.py",
               "--subsets", *subsets, "--n-clients-per-subset", str(n_per),
               "--seeds", *map(str, chunk), "--device", gpu,
               "--out-root", out_root]
        if skip_cells:
            cmd += ["--skip-cells", *skip_cells]
        logp = _logpath(out_root, gpu)
        print(f"{gpu}: seeds {chunk}  ->  {logp}", flush=True)
        procs.append(subprocess.Popen(
            cmd, cwd=REPO, stdout=open(logp, "w"), stderr=subprocess.STDOUT,
            start_new_session=True))
    print("Launched (detached, non-blocking). Now run  watch(procs, out_root).",
          flush=True)
    return procs


def tail_logs(out_root, n=25, gpus=("cuda:0", "cuda:1")):
    """One-shot: print the last n lines of each GPU log."""
    for gpu in gpus:
        logp = _logpath(out_root, gpu)
        print(f"\n===== {gpu}  ({logp}) =====", flush=True)
        if os.path.exists(logp):
            with open(logp, errors="ignore") as fh:
                lines = fh.readlines()
            print("".join(lines[-n:]).rstrip() or "(empty so far)")
        else:
            print("(no log yet)")


def watch(procs, out_root, every=20, n=18):
    """Live auto-refreshing tail until all procs finish. Interrupting this (⏹)
    only stops the watch — the detached workers keep training. Re-run to resume."""
    try:
        while True:
            running = sum(p.poll() is None for p in procs)
            clear_output(wait=True)
            print(f"[{time.strftime('%H:%M:%S')}]  {running}/{len(procs)} "
                  f"GPU workers running", flush=True)
            tail_logs(out_root, n=n)
            if running == 0:
                print("\nAll done. Return codes:", [p.returncode for p in procs])
                return
            time.sleep(every)
    except KeyboardInterrupt:
        print("\nStopped watching — training continues in the background. "
              "Re-run this cell to resume watching.")


def wait_all(procs):
    """Block until the parallel runs finish; prints their exit codes."""
    codes = [p.wait() for p in procs]
    print("Return codes:", codes, flush=True)
    return codes


## 7. Exp 2 — N=6 (FD001 + FD003, 3 + 3 clients)

Same data as the N=4 paper, more clients. **Krum-f2 becomes valid at N=6**
(n − f − 2 = 2 ≥ 1), so it's kept. Attacker placement is auto-resolved
(single = `client_4`, coordinated = `client_4` + `client_5`). ~a few hours on
one T4; roughly halved across both.


In [12]:
# Exp 2 — N=6, 5 seeds split across both T4s (Krum-f2 valid at N=6).
# Detached launch, then live watch. Pressing ⏹ stops the WATCH, not training.
procs = run_parallel(["FD001", "FD003"], n_per=3, out_root="results/rq7_n6_fd13")
watch(procs, "results/rq7_n6_fd13")

# One-GPU alternative (streams inline, but uses a single T4):
# run_campaign(["FD001", "FD003"], n_per=3, out_root="results/rq7_n6_fd13")


[10:40:59]  1/2 GPU workers running

===== cuda:0  (/kaggle/working/FL-for-Aircraft/results_rq7_n6_fd13_cuda0.log) =====

========= D23_gradscale_krum: grad ×-10 + Krum (f=1) =========
  attackers: client_4
round   1/50  lr=1.00e-03  loss=710.896  RMSE=70.57  NASA=211860  F1=0.431  delta_norms[client_1=3.95 client_2=3.84 client_3=3.93 client_4=44.41 client_5=4.43 client_6=4.36]  (2.0s)
round  10/50  lr=9.19e-04  loss=79.687  RMSE=21.21  NASA=4582  F1=0.805  delta_norms[client_1=1.30 client_2=0.82 client_3=1.11 client_4=16.50 client_5=1.81 client_6=1.41]  (1.9s)
round  20/50  lr=6.73e-04  loss=77.662  RMSE=22.92  NASA=4278  F1=0.773  delta_norms[client_1=1.08 client_2=0.38 client_3=0.72 client_4=11.99 client_5=1.22 client_6=0.88]  (1.9s)
round  30/50  lr=3.58e-04  loss=81.553  RMSE=23.62  NASA=3170  F1=0.789  delta_norms[client_1=0.82 client_2=0.21 client_3=0.50 client_4=7.19 client_5=0.74 client_6=0.56]  (2.1s)
round  40/50  lr=9.93e-05  loss=74.346  RMSE=18.18  NASA=1923  F1=0.899  de

In [ ]:
# Progress check — counts finished cells per seed by reading the per-round CSVs,
# so it's immune to log buffering. Self-contained; safe to re-run anytime.
import glob, os

REPO = "/kaggle/working/FL-for-Aircraft"


def progress(out_root, seeds=(42, 43, 44, 45, 46), cells=25):
    done = 0
    for s in seeds:
        d = f"{REPO}/{out_root}/seed_{s}"
        n = len(glob.glob(f"{d}/per_round_*.csv"))
        full = os.path.exists(f"{d}/metrics.json")
        done += n
        print(f"  seed {s:>2}: {n:>2}/{cells} cells" + ("  complete" if full else ""))
    tot = len(seeds) * cells
    print(f"Cells done: {done}/{tot}  ({100 * done // tot}%)")


progress("results/rq7_n6_fd13")


: 

## 8. Exp 3 — FD002 + FD004 (N=4, 2 + 2 clients)

The heavy one: ~5× the data of FD001/FD003, so a **separate bundle** (mixing all
four is blocked — different informative-sensor sets). `n_features=19` and the
backdoor trigger index (`s_3` → 5) are auto-detected; the CNN is unchanged.
**Krum-f2 is undefined at N=4**, so skip that cell. Use **both T4s** here.

> Caveat to disclose in the paper: normalisation is global, not regime-aware.


In [ ]:
# Exp 3 — FD002+FD004, 5 seeds split across both T4s. Skip Krum-f2 (undefined at N=4).
procs = run_parallel(["FD002", "FD004"], n_per=2, out_root="results/rq7_fd24",
                     skip_cells=["D54_coord_krum_f2"])
watch(procs, "results/rq7_fd24")

# Single-GPU alternative (slower):
# run_campaign(["FD002", "FD004"], n_per=2, out_root="results/rq7_fd24",
#              skip_cells=["D54_coord_krum_f2"])


## 9. Aggregate seeds & download — do the download BEFORE stopping!

Aggregates each campaign's `seed_*/metrics.json` into mean ± std, then zips the
whole `results/` tree. Download the zip from the Kaggle **Output / Data** panel
(right side) and unzip into your local repo's `results/` folder.


In [ ]:
import shutil, subprocess, sys, os

REPO = "/kaggle/working/FL-for-Aircraft"

for root in ["results/rq7_n6_fd13", "results/rq7_fd24"]:
    if os.path.isdir(f"{REPO}/{root}"):
        print(f"\n=== aggregating {root} ===", flush=True)
        # --out-file keeps each campaign's aggregate in its own folder;
        # otherwise both write to the same default and clobber each other.
        subprocess.run([sys.executable, "scripts/aggregate_rq7_seeds.py",
                        "--seeds-root", root,
                        "--out-file", f"{root}/metrics_aggregated.json"], cwd=REPO)

zip_path = shutil.make_archive("/kaggle/working/rq7_gpu_results", "zip",
                               f"{REPO}/results")
print("\nZipped ->", zip_path)
print("Download it from the Kaggle Output/Data panel, then unzip into your "
      "local repo's results/ folder.")
